# UBKI: компанії для допарсингу бенефіціарів
1. `Бенеф_1_ПІБ / Назва` порожній → бенефіціарів немає взагалі
2. серед `Бенеф_N_ПІБ / Назва` є запис без імені фізособи або юридична особа
3. сума часток бенефіціарів > 100 або частки не заповнені у 2+ бенефіціарів

Наприкінці зберігається очищений датафрейм без компаній з хоча б однією проблемою.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

# ================== ШЛЯХИ ==================
INPUT_PATH = r""    # <-- шлях до *_processed.csv (або .xlsx) після ubki_processing.ipynb
OUTPUT_DIR = r""    # <-- куди зберегти результати; порожнє = поруч з INPUT
# ===========================================

INPUT_PATH = Path(INPUT_PATH)
OUTPUT_DIR = Path(OUTPUT_DIR) if OUTPUT_DIR else INPUT_PATH.parent

if INPUT_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
else:
    df = pd.read_excel(INPUT_PATH, dtype=str).fillna("")

df["ЄДРПОУ"] = df["ЄДРПОУ"].str.strip().str.zfill(8)
N_TOTAL = len(df)

PREF, NAME_KEY, SHARE_KEY = "Бенеф", "ПІБ / Назва", "Частка"
BASE_COLS = [c for c in ["ЄДРПОУ", "Повна назва", "OPF_NAME", "STATUS", f"{PREF}_кількість"] if c in df.columns]

N_MAX = max(int(m.group(1)) for c in df.columns if (m := re.fullmatch(rf"{PREF}_(\d+)_{re.escape(NAME_KEY)}", c)))
name_cols = [f"{PREF}_{n}_{NAME_KEY}" for n in range(1, N_MAX + 1)]
share_cols = [f"{PREF}_{n}_{SHARE_KEY}" for n in range(1, N_MAX + 1)]
print(f"Компаній: {N_TOTAL}, колонок бенефіціарів: {N_MAX}")

## Класифікація записів у `Бенеф_N_ПІБ / Назва`
- **NO_NAME** — службовий текст замість імені («КІНЦЕВИЙ БЕНЕФІЦІАРНИЙ ВЛАСНИК ВІДСУТНІЙ», «ЧЛЕНИ ОРГАНІЗАЦІЇ», …)
- **LEGAL_ENTITY** — юридична особа (ТОВ, ПП, АТ, LIMITED, LTD, GMBH, лапки в назві, …)
- **NOT_PERSON_FORMAT** — не схоже на ПІБ (цифри, понад 5 слів тощо)
- **PERSON** — схоже на ПІБ фізособи

Списки ключових слів можна доповнити — після запуску нижче виводяться приклади кожної категорії.

In [ ]:
NO_NAME_RE = re.compile(
    r"БЕНЕФІЦІАР|ВІДСУТН|НЕ\s+БАЗУЄТЬСЯ|НЕ\s+ВСТАНОВЛ|НЕ\s+ВИЗНАЧ|НЕМАЄ|ЧЛЕН|УЧАСНИК|ДЕЛЕГАТ|"
    r"ТРУДОВИЙ\s+КОЛЕКТИВ|ФІЗИЧНІ\s+ОСОБИ|ЗАСНОВНИК|ПРАЦІВНИК",
    re.I,
)
LEGAL_RE = re.compile(
    r"ТОВАРИСТВ|ПІДПРИЄМСТВ|КОРПОРАЦ|КОМПАНІ|КОНЦЕРН|ХОЛДИНГ|ОБ['’ʼ]?ЄДНАННЯ|АСОЦІАЦ|СПІЛК|"
    r"ОРГАНІЗАЦ|ФОНД\b|ФОНДУ\b|БАНК\b|КООПЕРАТИВ|УСТАНОВ|\bДЕРЖАВ(А|И|І|У|НИЙ|НА|НЕ|НОГО|НОЇ|НОМУ|НІЙ)\b|МІНІСТЕРСТВ|РАДА\b|УПРАВЛІНН|ДЕПАРТАМЕНТ|"
    r"ГРОМАД|ЦЕНТР\b|ІНСТИТУТ|УНІВЕРСИТЕТ|ЦЕРКВ|ПАРТІ|ФЕДЕРАЦ|"
    r"\b(ТОВ|ПП|ПрАТ|ПАТ|АТ|ДП|КП|ТзОВ|ФГ|СГ|ВАТ|ЗАТ)\b|"
    r"\b(LIMITED|LTD|LLC|L\.L\.C|INC|CORP|CORPORATION|COMPANY|GMBH|AG|S\.?A\.?|B\.?V\.?|N\.?V\.?|"
    r"PLC|LP|LLP|SRL|S\.R\.L|SP\.?\s*Z\s*O\.?O|HOLDING|HOLDINGS|TRUST|FUND|FOUNDATION|GROUP|INVESTMENTS?)\b|"
    r"[\"«»“”]",
    re.I,
)
PERSON_RE = re.compile(r"^[A-Za-zА-ЩЬЮЯЄІЇҐа-щьюяєіїґ'’ʼ\-\.]+(\s+[A-Za-zА-ЩЬЮЯЄІЇҐа-щьюяєіїґ'’ʼ\-\.]+){1,4}$")


def classify(name: str) -> str:
    s = re.sub(r"\s+", " ", str(name)).strip()
    if not s:
        return ""
    if NO_NAME_RE.search(s):
        return "NO_NAME"
    if LEGAL_RE.search(s):
        return "LEGAL_ENTITY"
    if not PERSON_RE.match(s):
        return "NOT_PERSON_FORMAT"
    return "PERSON"


# довгий формат: ЄДРПОУ, №, ПІБ, частка, клас
long = (
    pd.DataFrame({
        "ЄДРПОУ": np.repeat(df["ЄДРПОУ"].to_numpy(), N_MAX),
        "№": np.tile(np.arange(1, N_MAX + 1), N_TOTAL),
        "ПІБ / Назва": df[name_cols].to_numpy().ravel(),
        "Частка": df[share_cols].to_numpy().ravel(),
    })
)
long = long[long["ПІБ / Назва"].str.strip() != ""].copy()
long["CLASS"] = long["ПІБ / Назва"].map(classify)
long["Частка"] = pd.to_numeric(long["Частка"].str.replace(",", "."), errors="coerce")

print(long["CLASS"].value_counts())
for cls in ["NO_NAME", "LEGAL_ENTITY", "NOT_PERSON_FORMAT"]:
    print(f"\n--- {cls} (топ-10) ---")
    print(long.loc[long["CLASS"] == cls, "ПІБ / Назва"].value_counts().head(10).to_string())

## 1. Немає жодного бенефіціара (`Бенеф_1_ПІБ / Назва` порожній)

In [ ]:
m1 = df[name_cols[0]].str.strip() == ""
df_p1 = df.loc[m1, BASE_COLS].copy()
df_p1["ПРОБЛЕМА"] = "NO_BENEFICIARY"
print(f"Проблема 1: {len(df_p1)} компаній")
df_p1.head()

## 2. Бенефіціар без імені фізособи або юридична особа

In [ ]:
bad = long[long["CLASS"].isin(["NO_NAME", "LEGAL_ENTITY", "NOT_PERSON_FORMAT"])]

# деталі: одна проблемна позиція на рядок
df_p2_details = bad.merge(df[BASE_COLS], on="ЄДРПОУ", how="left")[
    BASE_COLS + ["№", "ПІБ / Назва", "CLASS"]
]

# на рівні компанії
df_p2 = (
    bad.groupby("ЄДРПОУ")
    .agg(
        К_СТЬ_ПРОБЛЕМНИХ=("CLASS", "size"),
        NO_NAME=("CLASS", lambda s: (s == "NO_NAME").sum()),
        LEGAL_ENTITY=("CLASS", lambda s: (s == "LEGAL_ENTITY").sum()),
        NOT_PERSON_FORMAT=("CLASS", lambda s: (s == "NOT_PERSON_FORMAT").sum()),
    )
    .reset_index()
)
df_p2 = df[BASE_COLS].merge(df_p2, on="ЄДРПОУ", how="inner")
df_p2["ПРОБЛЕМА"] = "NOT_NATURAL_PERSON"

print(f"Проблема 2: {len(df_p2)} компаній, {len(bad)} проблемних записів")
print(bad["CLASS"].value_counts().to_string())
df_p2.head()

## 3. Сума часток > 100 або частки не заповнені у 2+ бенефіціарів
Службові записи (NO_NAME) не рахуються як бенефіціари.

In [ ]:
benef = long[long["CLASS"] != "NO_NAME"]
agg = (
    benef.groupby("ЄДРПОУ")
    .agg(
        К_СТЬ_БЕНЕФ=("ПІБ / Назва", "size"),
        СУМА_ЧАСТОК=("Частка", "sum"),
        К_СТЬ_БЕЗ_ЧАСТКИ=("Частка", lambda s: s.isna().sum()),
    )
    .reset_index()
)
agg["СУМА_ЧАСТОК"] = agg["СУМА_ЧАСТОК"].round(2)
agg["SUM_OVER_100"] = agg["СУМА_ЧАСТОК"] > 100.01
agg["MISSING_2PLUS"] = agg["К_СТЬ_БЕЗ_ЧАСТКИ"] >= 2

df_p3 = agg[agg["SUM_OVER_100"] | agg["MISSING_2PLUS"]]
df_p3 = df[BASE_COLS].merge(df_p3, on="ЄДРПОУ", how="inner")
df_p3["ПРОБЛЕМА"] = np.select(
    [df_p3["SUM_OVER_100"] & df_p3["MISSING_2PLUS"], df_p3["SUM_OVER_100"]],
    ["SUM_OVER_100 + MISSING_2PLUS", "SUM_OVER_100"],
    "MISSING_2PLUS",
)

print(f"Проблема 3: {len(df_p3)} компаній")
print(df_p3["ПРОБЛЕМА"].value_counts().to_string())
df_p3.head()

## Статистика

In [ ]:
ids = {
    "1. Немає бенефіціарів": set(df_p1["ЄДРПОУ"]),
    "2. Не фізособа / без імені": set(df_p2["ЄДРПОУ"]),
    "3. Частки (>100 або 2+ порожніх)": set(df_p3["ЄДРПОУ"]),
}
all_ids = set().union(*ids.values())

stats = pd.DataFrame(
    [(k, len(v), round(100 * len(v) / N_TOTAL, 2)) for k, v in ids.items()]
    + [("Разом унікальних ЄДРПОУ на допарсинг", len(all_ids), round(100 * len(all_ids) / N_TOTAL, 2))],
    columns=["Ситуація", "Компаній", "% від усіх"],
)
print(f"Усього компаній: {N_TOTAL}\n")
print(stats.to_string(index=False))

print("\nДеталізація:")
print(f"  2. проблемних записів: {len(bad)}")
for cls, n in bad["CLASS"].value_counts().items():
    print(f"     {cls}: {n} записів, {bad.loc[bad['CLASS'] == cls, 'ЄДРПОУ'].nunique()} компаній")
print(f"  3. сума > 100: {df_p3['SUM_OVER_100'].sum()} компаній")
print(f"     2+ без частки: {df_p3['MISSING_2PLUS'].sum()} компаній")
print(f"     обидві умови: {(df_p3['SUM_OVER_100'] & df_p3['MISSING_2PLUS']).sum()} компаній")

print("\nПеретини:")
print(f"  1 ∩ 2: {len(ids['1. Немає бенефіціарів'] & ids['2. Не фізособа / без імені'])}")
print(f"  1 ∩ 3: {len(ids['1. Немає бенефіціарів'] & ids['3. Частки (>100 або 2+ порожніх)'])}")
print(f"  2 ∩ 3: {len(ids['2. Не фізособа / без імені'] & ids['3. Частки (>100 або 2+ порожніх)'])}")
stats

In [ ]:
# спільний список на допарсинг з прапорцями
df_reparse = df[BASE_COLS].copy()
df_reparse["P1_NO_BENEFICIARY"] = df_reparse["ЄДРПОУ"].isin(ids["1. Немає бенефіціарів"]).astype(int)
df_reparse["P2_NOT_NATURAL_PERSON"] = df_reparse["ЄДРПОУ"].isin(ids["2. Не фізособа / без імені"]).astype(int)
df_reparse["P3_SHARES"] = df_reparse["ЄДРПОУ"].isin(ids["3. Частки (>100 або 2+ порожніх)"]).astype(int)
df_reparse = df_reparse[df_reparse[["P1_NO_BENEFICIARY", "P2_NOT_NATURAL_PERSON", "P3_SHARES"]].any(axis=1)]

if "STATUS" in df_reparse:
    print("Розподіл за статусом компанії:")
    print(df_reparse.groupby("STATUS")[["P1_NO_BENEFICIARY", "P2_NOT_NATURAL_PERSON", "P3_SHARES"]].sum())
print(len(df_reparse))

## Збереження

In [ ]:
out_xlsx = OUTPUT_DIR / "ubki_beneficiary_issues.xlsx"
sheets = {
    "reparse_all": df_reparse,
    "p1_no_benef": df_p1,
    "p2_not_person": df_p2,
    "p2_details": df_p2_details,
    "p3_shares": df_p3,
    "stats": stats,
}

try:
    import xlsxwriter  # noqa: F401
    engine = "xlsxwriter"
except ImportError:
    engine = "openpyxl"

with pd.ExcelWriter(out_xlsx, engine=engine) as w:
    for name, d in sheets.items():
        if len(d) >= 1_048_576:
            print(f"[!] {name}: {len(d)} рядків > ліміт Excel, лише CSV")
            continue
        d.to_excel(w, sheet_name=name, index=False)

out_csv = OUTPUT_DIR / "ubki_reparse_all.csv"
df_reparse.to_csv(out_csv, index=False, encoding="utf-8-sig")

print("XLSX ->", out_xlsx)
print("CSV  ->", out_csv)

# ---------- очищений датафрейм: без компаній з хоча б однією проблемою ----------
df_clean = df[~df["ЄДРПОУ"].isin(all_ids)].reset_index(drop=True)
print(f"\nОчищений датафрейм: {len(df_clean)} з {N_TOTAL} компаній "
      f"(видалено {N_TOTAL - len(df_clean)})")

clean_base = OUTPUT_DIR / (INPUT_PATH.stem + "_clean")
df_clean.to_csv(clean_base.with_suffix(".csv"), index=False, encoding="utf-8-sig")
print("CSV  ->", clean_base.with_suffix(".csv"))
if len(df_clean) < 1_048_576 and df_clean.shape[1] <= 16_384:
    df_clean.to_excel(clean_base.with_suffix(".xlsx"), index=False, engine=engine)
    print("XLSX ->", clean_base.with_suffix(".xlsx"))